# 01 · EDA-проверки и схемы валидации (R-L1-1)

Воспроизводит цифры из `docs/research/eda.md`. Регрессия ~30 с, классификация ~2.5 мин на 4 CPU.
Перед коммитом: `Clear All Outputs`.

In [ ]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import GroupKFold, KFold, StratifiedGroupKFold, StratifiedKFold

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
PUBLIC = ROOT / "tasks" / "public"
# Быстрые настройки для разведки — не путать с BASE_PARAMS агента
PARAMS = dict(
    n_estimators=2000,
    learning_rate=0.1,
    num_leaves=63,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
    n_jobs=-1,
)


def as_category(x: pd.DataFrame) -> pd.DataFrame:
    x = x.copy()
    for c in x.select_dtypes(exclude="number"):
        x[c] = x[c].astype("category")
    return x


def cv(model_cls, x, y, splits, score) -> str:
    """Прогоняет модель по готовым разбиениям и печатает среднее ± разброс."""
    scores = []
    for tr, va in splits:
        m = model_cls(**PARAMS).fit(
            x.iloc[tr], y[tr], eval_set=[(x.iloc[va], y[va])], callbacks=[lgb.early_stopping(100, verbose=False)]
        )
        scores.append(score(y[va], m.predict(x.iloc[va])))
    return f"{np.mean(scores):.4f} ± {np.std(scores):.4f}  фолды {[round(s, 3) for s in scores]}"

## 1. Регрессия: таргет, время, повторы зданий

In [ ]:
reg_tr = pd.read_csv(PUBLIC / "regression" / "train.csv")
reg_te = pd.read_csv(PUBLIC / "regression" / "test.csv")
y = reg_tr["site_eui"]
print(y.describe(percentiles=[0.01, 0.5, 0.99]).round(2).to_string())
print("skew", round(y.skew(), 2), "| skew log1p", round(np.log1p(y).skew(), 2), "| нулей", int((y <= 0).sum()))
print("Year_Factor train:", reg_tr["Year_Factor"].value_counts().sort_index().to_dict())
print("Year_Factor test: ", reg_te["Year_Factor"].value_counts().sort_index().to_dict())

In [ ]:
# В данных нет id здания — собираем ключ из признаков, которые не меняются по годам (⚠️ подобран руками)
STATIC = ["State_Factor", "building_class", "facility_type", "floor_area", "year_built", "ELEVATION"]


def building_key(df: pd.DataFrame) -> pd.Series:
    return df[STATIC].astype(str).agg("|".join, axis=1)


reg_tr["building"], reg_te["building"] = building_key(reg_tr), building_key(reg_te)
size = reg_tr.groupby("building").size()
print("зданий", reg_tr["building"].nunique(), "на", len(reg_tr), "строк")
print("доля строк в повторяющихся зданиях", round(reg_tr["building"].map(size).gt(1).mean(), 3))
print("доля зданий test, известных по train", round(reg_te["building"].isin(set(reg_tr["building"])).mean(), 3))
print(
    "std таргета внутри здания (медиана)",
    round(reg_tr.groupby("building")["site_eui"].std().median(), 2),
    "| общий",
    round(y.std(), 2),
)

In [ ]:
x = as_category(reg_tr.drop(columns=["id", "site_eui", "building"]))
yv = y.to_numpy()
year = reg_tr["Year_Factor"].to_numpy()


def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


print("KFold(5, shuffle)      ", cv(lgb.LGBMRegressor, x, yv, KFold(5, shuffle=True, random_state=42).split(x), rmse))
print("GroupKFold(5, здание)  ", cv(lgb.LGBMRegressor, x, yv, GroupKFold(5).split(x, groups=reg_tr["building"]), rmse))
print("Время: 1–4 → 5         ", cv(lgb.LGBMRegressor, x, yv, [(np.where(year < 5)[0], np.where(year == 5)[0])], rmse))
expanding = [(np.where(year <= k)[0], np.where(year == k + 1)[0]) for k in (2, 3, 4)]
print("Время: expanding       ", cv(lgb.LGBMRegressor, x, yv, expanding, rmse))

## 2. Классификация: дисбаланс, категории, дубликаты

In [ ]:
mc_tr = pd.read_csv(PUBLIC / "multiclass" / "train.csv")
mc_te = pd.read_csv(PUBLIC / "multiclass" / "test.csv")
TARGET = "attack_cat"
feats = [c for c in mc_tr.columns if c not in ("id", TARGET)]

share = mc_tr[TARGET].value_counts(normalize=True)
print("частый / редкий =", round(share.iloc[0] / share.iloc[-1]), "\n", share.round(4).to_string())
top = mc_tr[feats].apply(lambda s: s.value_counts(normalize=True, dropna=False).iloc[0])
print("константы и квазиконстанты (>99%):", top[top > 0.99].round(4).to_dict())
for c in mc_tr[feats].select_dtypes(exclude="number"):
    vc = mc_tr[c].value_counts()
    print(
        f"{c}: уникальных {len(vc)}, редких(<10) {(vc < 10).sum()}, новые в test {sorted(set(mc_te[c]) - set(mc_tr[c]))}"
    )

In [ ]:
h_tr = pd.util.hash_pandas_object(mc_tr[feats], index=False)
h_te = pd.util.hash_pandas_object(mc_te[feats], index=False)
dup = h_tr.duplicated(keep=False)
conflict = h_tr.map(mc_tr.groupby(h_tr)[TARGET].nunique()) > 1
print("строк в группах дубликатов", round(dup.mean(), 3), "| групп", h_tr[dup].nunique())
print("строк-дубликатов с разными метками", round(conflict.mean(), 3))
print("доля дубликатов по классам", mc_tr.assign(d=dup).groupby(TARGET)["d"].mean().round(2).to_dict())
print("строк test, точно совпадающих с train", round(h_te.isin(set(h_tr)).mean(), 3))

In [ ]:
x = as_category(mc_tr[feats])
yc = mc_tr[TARGET].astype("category").cat.codes.to_numpy()


def macro_f1(a, b):
    return f1_score(a, b, average="macro")


print(
    "KFold(5)               ", cv(lgb.LGBMClassifier, x, yc, KFold(5, shuffle=True, random_state=42).split(x), macro_f1)
)
print(
    "StratifiedKFold(5)     ",
    cv(lgb.LGBMClassifier, x, yc, StratifiedKFold(5, shuffle=True, random_state=42).split(x, yc), macro_f1),
)
sgkf = StratifiedGroupKFold(5, shuffle=True, random_state=42).split(x, yc, groups=h_tr.to_numpy())
print("StratifiedGroupKFold(5)", cv(lgb.LGBMClassifier, x, yc, sgkf, macro_f1))

Выводы и список проверок для агента — в `docs/research/eda.md`.